(projections)=
# Projection Images & Cellpose Inputs

```{toctree}
:maxdepth: 2
```

This guide explains all the intermediate projection images computed during Suite2p processing and how they are used as inputs to Cellpose anatomical segmentation.

```{tip}
Understanding these images helps you choose the best `anatomical_only` mode and tune `spatial_hp_cp` for your data.
```

## Overview

Suite2p computes several reference images during registration and detection. These images serve different purposes:

| Image | Computed From | Primary Use |
|-------|--------------|-------------|
| `meanImg` | Mean of registered movie | Registration reference |
| `meanImgE` | Enhanced mean (HP filtered) | Anatomical detection (mode 3) |
| `max_proj` | Max of HP-filtered movie | Anatomical detection (mode 4) |
| `refImg` | Initial frames (pre-reg) | Registration template |
| `Vcorr` | Pixel correlation map | Functional detection |

## Suite2p Output Images

After registration completes, Suite2p stores these images in the `ops` dictionary:

```{figure} _images/projections/01_raw_projections.png
:alt: Suite2p output images
:name: proj-fig-raw
:width: 100%

**Suite2p reference images** stored in `ops.npy`. From left to right: mean image (`meanImg`), enhanced mean (`meanImgE`), maximum projection (`max_proj`), registration reference (`refImg`), and correlation map (`Vcorr`).
```

### How Each Image is Computed

**meanImg**: Simple temporal mean of the registered movie
```python
meanImg = registered_movie.mean(axis=0)
```

**meanImgE**: Enhanced mean with spatial high-pass filter to sharpen cell boundaries
```python
from scipy.ndimage import median_filter

def enhanced_mean(mean_img, diameter=12):
    I = mean_img.astype(np.float32)
    d = int(4 * np.ceil(diameter) + 1)
    Imed = median_filter(I, size=d)
    I = I - Imed  # high-pass: subtract local median
    Idiv = median_filter(np.abs(I), size=d)
    I = I / (1e-10 + Idiv)  # normalize by local contrast
    return np.clip((I + 6) / 12, 0, 1)
```

**max_proj**: Maximum projection of the temporally high-pass filtered movie
```python
# after temporal HP filtering removes slow baseline
max_proj = hp_filtered_movie.max(axis=0)
```

**refImg**: Built from the first N frames during registration initialization
```python
# uses ops['nimg_init'] frames to build template
refImg = build_reference(movie[:nimg_init])
```

**Vcorr**: Local pixel correlation, high values indicate coordinated activity
```python
# correlation of each pixel with its neighbors
Vcorr = local_correlation_map(hp_filtered_movie)
```

## Cellpose Input Modes (`anatomical_only`)

The `anatomical_only` parameter controls which image is fed to Cellpose for anatomical segmentation:

| Value | Input Image | Best For |
|-------|-------------|----------|
| `0` | None (functional detection) | Activity-based ROIs |
| `1` | `max_proj / meanImg` | Highlighting active cells |
| `2` | `meanImg` | Structural features only |
| `3` | `meanImgE` | **Recommended**: clear boundaries |
| `4` | `max_proj` | Brightest activity regions |

```{figure} _images/projections/02_anatomical_modes.png
:alt: Anatomical detection modes
:name: proj-fig-modes
:width: 100%

**Cellpose input images** for each `anatomical_only` mode (1-4). Mode 3 (`meanImgE`) typically provides the best cell boundary definition.
```

### Choosing the Right Mode

**Mode 3 (`meanImgE`)** - Recommended for most datasets:
- Spatial high-pass filtering enhances cell boundaries
- Works well for densely labeled tissue
- Less sensitive to brightness variations

**Mode 4 (`max_proj`)** - Use when:
- Cells have strong transient activity
- Mean image is too dim or noisy
- You want to prioritize active cells

**Mode 1 (`max_proj / meanImg`)** - Use when:
- You want to emphasize cells with high activity relative to baseline
- Background is relatively uniform

**Mode 2 (`meanImg`)** - Use when:
- Raw structural features are clear
- Enhancement artifacts are problematic

## Spatial High-Pass Filter (`spatial_hp_cp`)

Before passing to Cellpose, an additional spatial high-pass filter can be applied via `spatial_hp_cp` (default: 0). This parameter ranges from 0 to 1:

- **0.0**: No filtering (use image as-is)
- **0.5**: Moderate high-pass (reduces background)
- **1.0**: Strong high-pass (maximum edge enhancement)

```{figure} _images/projections/03_spatial_hp_filter.png
:alt: Spatial high-pass filter effect
:name: proj-fig-spatial-hp
:width: 100%

**Effect of `spatial_hp_cp`** on the Cellpose input image. Higher values increase edge contrast but may introduce noise in dim regions.
```

### When to Use Spatial HP Filtering

**Use higher values (0.3-0.7)** when:
- Cell boundaries are unclear in the raw image
- There's significant background fluorescence
- Cells are closely packed

**Keep at 0** when:
- `meanImgE` already has good contrast
- Image is noisy (HP can amplify noise)
- Cell boundaries are already well-defined

## Final Cellpose Input

The complete pipeline for generating Cellpose input:

1. **Registration** → produces `meanImg`, `refImg`
2. **Enhancement** → computes `meanImgE` from `meanImg`
3. **Detection preprocessing** → computes `max_proj` from HP-filtered movie
4. **Mode selection** → picks image based on `anatomical_only`
5. **HP filtering** → applies `spatial_hp_cp` if > 0

```{figure} _images/projections/04_cellpose_final_input.png
:alt: Final Cellpose input comparison
:name: proj-fig-final
:width: 100%

**What Cellpose actually receives** under different parameter combinations. Top row: `spatial_hp_cp=0`. Bottom row: with `spatial_hp_cp` applied.
```

## Detail Comparison

Zoomed view showing how spatial filtering affects cell boundary definition:

```{figure} _images/projections/05_hp_filter_zoom.png
:alt: High-pass filter detail
:name: proj-fig-zoom
:width: 100%

**Zoomed comparison** of cell boundaries with and without spatial high-pass filtering. The filter sharpens edges but can introduce artifacts around bright structures.
```

## Accessing Projection Images

After running the pipeline, you can access all projection images from the `ops` dictionary:

```python
import numpy as np
import matplotlib.pyplot as plt
from lbm_suite2p_python import load_ops

# load ops from a processed plane
ops = load_ops("path/to/plane0/ops.npy")

# access projection images
mean_img = ops["meanImg"]      # temporal mean
mean_imgE = ops["meanImgE"]    # enhanced mean
max_proj = ops["max_proj"]     # max projection
ref_img = ops["refImg"]        # registration reference
vcorr = ops.get("Vcorr")       # correlation map (may be None)

# note: images may be cropped relative to original
# use yrange/xrange to match coordinates
yrange = ops["yrange"]
xrange = ops["xrange"]
print(f"Image shape: {mean_img.shape}")
print(f"Crop: y={yrange}, x={xrange}")

# plot all projections
fig, axes = plt.subplots(1, 5, figsize=(20, 4))
for ax, (img, title) in zip(axes, [
    (mean_img, "meanImg"),
    (mean_imgE, "meanImgE"),
    (max_proj, "max_proj"),
    (ref_img, "refImg"),
    (vcorr, "Vcorr"),
]):
    if img is not None:
        ax.imshow(img, cmap="gray")
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()
```

## Recommendations

For most LBM datasets:

```python
ops = {
    "anatomical_only": 3,      # use enhanced mean
    "spatial_hp_cp": 0.0,      # usually not needed with meanImgE
    "diameter": 6,             # adjust to your cell size in pixels
    "cellprob_threshold": 0.0, # standard threshold
    "flow_threshold": 0.4,     # permissive for LBM
}
```

If cells aren't well detected:
1. Try `anatomical_only=4` (max projection)
2. Increase `spatial_hp_cp` to 0.3-0.5
3. Adjust `diameter` to match your cell sizes
4. Lower `cellprob_threshold` to -1 or -2

## See Also

- {doc}`User Guide <user_guide>` - Complete parameter reference
- {doc}`Image Gallery <image_gallery>` - Visual reference for all outputs
- [Cellpose Documentation](https://cellpose.readthedocs.io/) - Cellpose model details